In [0]:
dbutils.widgets.text("eventhub_namespace", "evhua5816bd", "Event Hub namespace")
dbutils.widgets.text("eventhub_name", "elina-sharabura-eventhub", "Event Hub name")
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd", "Catalog")
dbutils.widgets.text("schema", "elina_sharabura", "Schema")

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("eventhub_consumer")

In [0]:
listen_connection_str = dbutils.secrets.get(
    scope="elina_sharabura-personal-scope",
    key="elina-sharabura-eventhub-connstr-listen"
)

EVENTHUB_NAMESPACE = dbutils.widgets.get("eventhub_namespace")
EVENTHUB_NAME = dbutils.widgets.get("eventhub_name")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")

bootstrap_servers = f"{EVENTHUB_NAMESPACE}.servicebus.windows.net:9093"

jaas_config = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="$ConnectionString" '
    f'password="{listen_connection_str}";'
)

kafka_options = {
    "kafka.bootstrap.servers": bootstrap_servers,
    "subscribe": EVENTHUB_NAME,
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": jaas_config,
    "kafka.group.id": "$Default",
    "startingOffsets": "earliest",
}

In [0]:
from pyspark.sql.types import StructType, StringType, LongType, BooleanType
from pyspark.sql.functions import col, from_json, current_timestamp

CHECKPOINT_PATH_EVENTHUB = f"/Volumes/{CATALOG}/{SCHEMA}/streaming_source/_checkpoints/eventhub/wikipedia"
TARGET_TABLE_EVENTHUB = f"{CATALOG}.{SCHEMA}_bronze.wikipedia_events"

wiki_schema = (
    StructType()
        .add("title", StringType())
        .add("wiki", StringType())
        .add("type", StringType())
        .add("user", StringType())
        .add("bot", BooleanType())
        .add("timestamp", LongType())
)

df_raw = spark.readStream.format("kafka").options(**kafka_options).load()

df_parsed = (
    df_raw
        .selectExpr("CAST(value AS STRING) as json_str")
        .withColumn("data", from_json(col("json_str"), wiki_schema))
        .select("data.*")
        .withColumn("ingestion_timestamp", current_timestamp())
)

query_wiki = (
    df_parsed.writeStream
        .format("delta")
        .option("checkpointLocation", CHECKPOINT_PATH_EVENTHUB)
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(TARGET_TABLE_EVENTHUB)
)

query_wiki.awaitTermination()

In [0]:
before_count = spark.table(TARGET_TABLE_EVENTHUB).count()

reload_query = (
    df_parsed.writeStream
        .format("delta")
        .option("checkpointLocation", CHECKPOINT_PATH_EVENTHUB)
        .trigger(availableNow=True)
        .toTable(TARGET_TABLE_EVENTHUB)
)
reload_query.awaitTermination()

after_count = spark.table(TARGET_TABLE_EVENTHUB).count()

logger.info(f"Before: {before_count}")
logger.info(f"After:  {after_count}")